# Modelo Bayesiano de Presencia-Background para el Copetón (Colab Version)
Este notebook implementa la regresión logística Bayesiana según lo documentado en `presence_only_logic.md`.

## 0. Instalación y Configuración para Google Colab
Ejecuta esta celda si estás en Colab para instalar las versiones necesarias y montar tu Drive si los datos están allí.

In [ ]:
# Instalación de PyMC y ArviZ (Colab suele traer versiones anteriores)
# !pip install pymc arviz

from google.colab import drive
import os

# Montar Google Drive
# drive.mount('/content/drive')

print("Asegúrate de subir el archivo 'copeton_presence_only_ready.csv' a la carpeta de Colab o a tu Drive.")

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

## 1. Carga de Datos
Cargamos los datos preprocesados de presencia y fondo (background). 

**Nota:** Si subes el archivo directamente al panel izquierdo de Colab, la ruta será `'copeton_presence_only_ready.csv'`.

In [ ]:
# Cambia esta ruta según dónde guardes el archivo en Colab
DATA_PATH = 'copeton_presence_only_ready.csv' 

if not os.path.exists(DATA_PATH):
    print(f"¡ERROR! No se encontró el archivo en {DATA_PATH}. Súbelo a Colab.")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"Datos cargados exitosamente: {df.shape}")
    display(df.head())

## 2. Preprocesamiento: Estandarización de Covariables
Estandarizamos los contaminantes para mejorar el muestreo de PyMC.

In [ ]:
# Definir covariables a usar
covariates = ['pm10_ugm3', 'pm25_ugm3', 'no2_ppb', 'o3_ppb', 'co_ppm', 'so2_ugm3']

# Limpiar NaNs para correr el modelo
df_clean = df.dropna(subset=covariates + ['y']).copy()

# Guardamos medias y desviaciones para recuperar Odds Ratios después
means = df_clean[covariates].mean()
stds = df_clean[covariates].std()

# Estandarizar
X_std = (df_clean[covariates] - means) / stds
y_obs = df_clean['y'].values

print(f'Datos listos: {len(df_clean)} registros (Presencias: {int(y_obs.sum())}, Fondo: {len(y_obs) - int(y_obs.sum())})')

## 3. Modelo Bayesiano en PyMC
Implementamos la Regresión Logística Presence-Background.

In [ ]:
with pm.Model() as bayesian_pb_model:
    
    # Datos observados
    X_data = pm.MutableData('X', X_std.values)
    
    # Priors débilmente informativos
    beta = pm.Normal('beta', mu=0, sigma=1, shape=X_std.shape[1])
    beta_0 = pm.Normal('beta_0', mu=0, sigma=2)
    
    # Predictor lineal
    eta = beta_0 + pm.math.dot(X_data, beta)
    
    # Función de enlace Logit
    psi = pm.math.invlogit(eta)
    
    # Likelihood
    obs = pm.Bernoulli('obs', p=psi, observed=y_obs)
    
pm.model_to_graphviz(bayesian_pb_model)

## 4. Muestreo MCMC (NUTS)
Corremos las cadenas.

In [ ]:
with bayesian_pb_model:
    # Muestreo
    trace = pm.sample(
        draws=1500, 
        tune=1000, 
        chains=2, # Reducido para mayor velocidad en Colab
        target_accept=0.9, 
        return_inferencedata=True
    )

## 5. Diagnósticos y Resultados

In [ ]:
az.plot_trace(trace)
plt.tight_layout()
plt.show()

In [ ]:
az.summary(trace)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
az.plot_forest(trace, var_names=['beta'], combined=True, ax=ax)
ax.set_yticklabels(covariates[::-1])
ax.axvline(0, color='red', linestyle='--')
plt.title('Efectos Estandarizados (Betas)')
plt.show()

## 6. Interpretación Ecológica (Odds Ratios)

In [ ]:
summary = az.summary(trace, var_names=['beta'])
beta_std_means = summary['mean'].values
beta_orig_means = beta_std_means / stds.values
odds_ratios = np.exp(beta_orig_means)

or_df = pd.DataFrame({
    'Contaminante': covariates,
    'Beta_Std': beta_std_means,
    'Beta_Original': beta_orig_means,
    'Odds_Ratio': odds_ratios
})

or_df